# 07 — Robustness to simulated haze (optional, no retraining)

**Purpose:** Probe whether the RGB/13-band advantage survives **synthetic atmospheric-looking perturbations**. For each available trained Random Forest, SmallCNN and ResNet18 checkpoint, evaluate the *unchanged held-out test set* under **uniform** or **wavelength-dependent** haze at fixed intensities; no model is retrained. Predictions and plots are derived from the real checkpoints and original GeoTIFF/cache data.

**Requires:** the complete preprocessing outputs from `02`, the test CSVs and local model files from `03`, `04`, `05`. Notebook `04` saves `models/cnn_<input>_seed<n>.pt`; notebook `03` saves `models/rf_<input>_seed<n>.joblib`; notebook `05` saves `models/resnet_<input>_seed<n>.pt`. Missing models are **skipped with a clear message**, never assigned invented performance. For final robustness of every model, use all three seeds.

**IMPORTANT:** The haze transform below is a **heuristic stress test**, not a radiative-transfer model nor physically calibrated Sentinel-2 simulation. No real cloud/haze labels are available in EuroSAT. Values mix raw pixel values with training-derived bright-reference band values. A positive or negative outcome must be described as *robustness to this synthetic perturbation only*, never real atmospheric performance.

**Run time:** Evaluation of many DL checkpoints can still take time, but does **not** retrain any model. Run after notebook 04/05 have finished. Start with `RUN_SEEDS=(0,)`; for a complete analysis change to `(0,1,2)`. Set `MAX_TEST_SAMPLES=256` for a quick diagnostic only; the final report should use `None` to evaluate the entire held-out test set.

## 1. Locate data, model weights and test predictions

No fixed Mac user paths, no complete cache materialization into RAM. We check the train-only band statistics and get the canonical rows of the test split.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'src' / 'piece_of_crop' / 'data.py').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Open the cloned piece_of_crop repository in VS Code (File > Open Folder).')

project_root = find_project_root()
src_dir = str(project_root / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
results_dir = project_root / 'results'
predictions_dir = results_dir / 'predictions'
figures_dir = results_dir / 'figures'
models_dir = project_root / 'models'
for directory in (results_dir, predictions_dir, figures_dir, models_dir):
    directory.mkdir(parents=True, exist_ok=True)
print('Project root:', project_root)

import contextlib
import io
import joblib
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score
from piece_of_crop.data import CLASS_NAMES, read_split, get_cache, normalise, select_bands

RUN_SEEDS = (0,)  # change to (0,1,2) for complete optional robustness analysis
STRENGTHS = (0.0, 0.15, 0.30)
PROFILES = ('uniform', 'wavelength')
MAX_TEST_SAMPLES = None  # set 256 for quick diagnostics; None for final full test
BATCH_SIZE = 32
SAVE_PREDICTIONS = True

split = read_split(results_dir/'split.csv')
stat_file = results_dir/'band_stats.csv'
if not stat_file.exists():
    raise FileNotFoundError('Run notebook 02 first: band_stats.csv is missing.')
band_stats = pd.read_csv(stat_file)
cache = get_cache(split, project_root/'data'/'EuroSAT_MS',project_root/'data'/'eurosat_cache.npy')
all_test = split.loc[split['split'].eq('test')].reset_index(drop=True)
test_rows = all_test.iloc[:MAX_TEST_SAMPLES] if MAX_TEST_SAMPLES is not None else all_test
if not len(test_rows):
    raise ValueError('Test set is empty.')
label_ids = np.arange(len(CLASS_NAMES),dtype=int)
assert cache.shape[0]==len(split)

if torch.cuda.is_available():
    device=torch.device('cuda')
elif hasattr(torch.backends,'mps') and torch.backends.mps.is_available():
    device=torch.device('mps')
else:
    device=torch.device('cpu')
print('Device:',device, '| test patches:',len(test_rows),'/',len(all_test))

# Import the exact RF feature extraction routine used by notebook 03.
feature_dir = str(project_root/'src'/'piece_of_crop')
if feature_dir not in sys.path:
    sys.path.insert(0,feature_dir)
from features import extract_rf_features

## 2. Define synthetic haze scenarios

For strength `a`, transform **raw** image bands: `x_hazy = (1 - a_band) * x + a_band * bright_reference_band`, where `bright_reference_band` is the **99.9th percentile clipping limit computed on training data** in notebook 02. `uniform`: same `a` for all bands; `wavelength`: hand-set multipliers favour larger perturbations in short visible wavelengths. These are transparent heuristics, not physically measured haze radiances. Normalization is applied **after** the perturbation using the original training statistics only.

In [ ]:
# Project band order: B01,B02,B03,B04,B05,B06,B07,B08,B09,B10,B11,B12,B8A.
# Unitless *heuristic* intensity coefficients, not measured optical-depth values.
WAVELENGTH_FACTORS=np.array([1.25,1.20,1.12,1.00,.88,.80,.75,.68,.58,.58,.52,.48,.70], dtype=np.float32)
bright_reference=band_stats['clip_high'].to_numpy(dtype=np.float32)
assert bright_reference.shape==(13,) and np.isfinite(bright_reference).all()

def simulate_haze(raw, strength, profile):
    raw=np.asarray(raw,dtype=np.float32)
    if raw.shape[-3]!=13:
        raise ValueError('Expected original 13-band EuroSAT patches')
    if not (0<=strength<=1):
        raise ValueError('Haze strength must lie in [0,1]')
    if profile=='uniform':
        factors=np.ones(13,dtype=np.float32)
    elif profile=='wavelength':
        factors=WAVELENGTH_FACTORS
    else:
        raise ValueError('Unknown haze profile')
    alpha=np.clip(strength*factors,0,1)
    # Broadcasting works for one (13,H,W) or a batch (N,13,H,W).
    leading = (1,)*(raw.ndim-3)
    alpha=alpha.reshape(*leading,13,1,1)
    air=bright_reference.reshape(*leading,13,1,1)
    return (1-alpha)*raw + alpha*air

example=np.asarray(cache[int(test_rows.iloc[0]['cache_row'])],dtype=np.float32)
assert np.array_equal(example,simulate_haze(example,0.,'uniform'))
assert np.array_equal(example,simulate_haze(example,0.,'wavelength'))
print('Haze=0 identity tests passed. Scenario strengths:',STRENGTHS)

## 3. Recreate the saved architectures and build deterministic inference paths

The SmallCNN architecture below **exactly matches notebook 04**, and the ResNet18 definition matches notebook 05 (same `conv1` shape and 10-class head). Checkpoints are loaded with `weights_only=True` from your own notebooks; no pretrained ImageNet download is needed for inference. We verify that the **zero-haze predictions match the saved clean test CSV** before accepting any haze result.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self,in_channels,num_classes=10):
        super().__init__()
        blocks=[]
        c=in_channels
        for n in (32,64,128):
            blocks.extend([nn.Conv2d(c,n,kernel_size=3,padding=1),nn.BatchNorm2d(n),
                           nn.ReLU(inplace=True),nn.MaxPool2d(kernel_size=2)])
            c=n
        self.features=nn.Sequential(*blocks)
        self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(128,num_classes))
    def forward(self,x):
        return self.head(self.features(x))


def load_deep_model(family,input_type,seed):
    path=models_dir/f'{family}_{input_type}_seed{seed}.pt'
    checkpoint=torch.load(path,map_location='cpu',weights_only=True)
    if checkpoint.get('input')!=input_type or int(checkpoint.get('seed',-1))!=seed:
        raise ValueError(f'Wrong checkpoint metadata in {path}')
    channels=3 if input_type=='rgb' else 13
    if family=='cnn':
        net=SmallCNN(channels,len(CLASS_NAMES))
        image_size=64
    elif family=='resnet':
        try:
            from torchvision.models import resnet18
        except Exception as exc:
            raise ImportError('ResNet evaluation needs torchvision; see environment.yml') from exc
        net=resnet18(weights=None)
        if channels==13:
            old=net.conv1
            net.conv1=nn.Conv2d(13,old.out_channels,kernel_size=old.kernel_size,
                                stride=old.stride,padding=old.padding,bias=False)
        net.fc=nn.Linear(net.fc.in_features,len(CLASS_NAMES))
        image_size=int(checkpoint['input_size'])
    else:
        raise ValueError(f'Unsupported DL family {family}')
    net.load_state_dict(checkpoint['state_dict'], strict=True)
    net=net.to(device).eval()
    return net,image_size


class HazyTestDataset(Dataset):
    def __init__(self,rows,cache,stats,input_type,profile,strength):
        self.rows=rows
        self.cache=cache
        self.stats=stats
        self.input_type=input_type
        self.profile=profile
        self.strength=strength
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,index):
        row=self.rows.iloc[index]
        raw=np.asarray(self.cache[int(row['cache_row'])],dtype=np.float32)
        hazy=simulate_haze(raw,self.strength,self.profile)
        x=select_bands(normalise(hazy,self.stats),self.input_type)
        return torch.from_numpy(np.ascontiguousarray(x)),int(row['label'])


@torch.inference_mode()
def predict_deep(net,image_size,input_type,profile,strength):
    dataset=HazyTestDataset(test_rows,cache,band_stats,input_type,profile,strength)
    loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    all_y,all_prob=[],[]
    for x,y in loader:
        x=x.to(device=device,dtype=torch.float32)
        if image_size!=64:
            x=F.interpolate(x,size=(image_size,image_size),mode='bilinear',align_corners=False)
        probabilities=torch.softmax(net(x),dim=1).cpu().numpy()
        all_y.append(y.numpy())
        all_prob.append(probabilities)
    y_true=np.concatenate(all_y)
    proba=np.concatenate(all_prob)
    assert np.array_equal(y_true,test_rows['label'].to_numpy(dtype=int))
    return proba


def predict_rf(model,input_type,profile,strength):
    prob_list=[]
    for start in range(0,len(test_rows),BATCH_SIZE):
        part=test_rows.iloc[start:start+BATCH_SIZE]
        idx=part['cache_row'].to_numpy(dtype=int)
        # Extract only one batch of images from the mmap each time.
        raw=np.asarray(cache[idx],dtype=np.float32)
        hazy=simulate_haze(raw,strength,profile)
        with contextlib.redirect_stdout(io.StringIO()):
            X_rgb,X_ms,_,_=extract_rf_features(hazy,band_stats,batch_size=BATCH_SIZE)
        X=X_rgb if input_type=='rgb' else X_ms
        raw_prob=model.predict_proba(X)
        proba=np.zeros((len(part),len(CLASS_NAMES)),dtype=float)
        for j,label in enumerate(model.classes_):
            proba[:,int(label)] = raw_prob[:,j]
        prob_list.append(proba)
    return np.vstack(prob_list)

## 4. Evaluate each available saved checkpoint, no retraining

A zero-haze sanity check **must reproduce clean predictions** for each model, input and seed; otherwise the model/haze result is not comparable. To save time while checking code, set `MAX_TEST_SAMPLES=256`. Use `None` and all three seeds for a full robustness table. Saved output names have `haze_` prefix so notebook 06 will not accidentally treat haze predictions as clean test predictions.

In [ ]:
family_prefixes={'RandomForest':'rf','SmallCNN':'cnn','ResNet18':'resnet'}
rows=[]
missing=[]
for model_name,prefix in family_prefixes.items():
    for input_type in ('rgb','13band'):
        for seed in RUN_SEEDS:
            extension='.joblib' if prefix=='rf' else '.pt'
            checkpoint_path=models_dir/f'{prefix}_{input_type}_seed{seed}{extension}'
            clean_file=predictions_dir/f'{prefix}_{input_type}_seed{seed}.csv'
            if not checkpoint_path.is_file() or not clean_file.is_file():
                missing.append((model_name,input_type,seed))
                print('Skipping missing weights or predictions:',model_name,input_type,seed)
                continue
            reference=pd.read_csv(clean_file)
            required={'file','y_true','y_pred'}
            if not required.issubset(reference.columns):
                raise ValueError(f'{clean_file.name} is missing {required-set(reference.columns)}')
            reference=test_rows[['file','label']].merge(
                reference[['file','y_true','y_pred']],on='file',validate='one_to_one',sort=False)
            if len(reference)!=len(test_rows) or not np.array_equal(reference['label'],reference['y_true']):
                raise ValueError(f'Clean predictions not aligned: {clean_file.name}')

            print(f'\n== {model_name}, {input_type}, seed {seed} ==',flush=True)
            if prefix=='rf':
                model=joblib.load(checkpoint_path)
                image_size=None
            else:
                model,image_size=load_deep_model(prefix,input_type,seed)

            for profile in PROFILES:
                for strength in STRENGTHS:
                    if prefix=='rf':
                        prob=predict_rf(model,input_type,profile,strength)
                    else:
                        prob=predict_deep(model,image_size,input_type,profile,strength)
                    pred=prob.argmax(axis=1)
                    if strength==0.:
                        matches=np.mean(pred==reference['y_pred'].to_numpy(dtype=int))
                        if matches!=1.:
                            raise ValueError(f'Zero-haze predictions do not reproduce clean model '
                                             f'{model_name}/{input_type}/seed{seed} '
                                             f'(agreement {matches:.3f}). Stop and check checkpoint/preprocessing.')
                    y_true=test_rows['label'].to_numpy(dtype=int)
                    row={'model':model_name,'input':input_type,'seed':seed,
                         'profile':profile,'strength':strength,'n_test':len(test_rows),
                         'accuracy':accuracy_score(y_true,pred),
                         'macro_f1':f1_score(y_true,pred,labels=label_ids,
                                             average='macro',zero_division=0),
                         'subset':len(test_rows)!=len(all_test)}
                    rows.append(row)
                    print(f'{profile} / a={strength:.2f} | acc={row["accuracy"]:.4f} '
                          f'F1={row["macro_f1"]:.4f}',flush=True)
                    if SAVE_PREDICTIONS:
                        out=pd.DataFrame({'file':test_rows['file'].to_numpy(),
                                          'y_true':y_true,'y_pred':pred,
                                          'confidence':prob.max(axis=1)})
                        out.to_csv(predictions_dir/
                                   f'haze_{prefix}_{input_type}_seed{seed}_{profile}_a{int(round(strength*100)):02d}.csv',
                                   index=False)
            del model
            if device.type=='mps' and hasattr(torch,'mps'):
                torch.mps.empty_cache()

if not rows:
    raise FileNotFoundError('No matching clean prediction CSV + model checkpoints. Run 03/04/05 first.')
haze_results=pd.DataFrame(rows)
# Delta is relative to the corresponding unperturbed scenario for that seed and profile.
base=haze_results.loc[haze_results['strength']==0,
                      ['model','input','seed','profile','accuracy','macro_f1']]
base=base.rename(columns={'accuracy':'clean_accuracy','macro_f1':'clean_macro_f1'})
haze_results=haze_results.merge(base,on=['model','input','seed','profile'],validate='many_to_one')
haze_results['accuracy_drop']=haze_results['clean_accuracy']-haze_results['accuracy']
haze_results['macro_f1_drop']=haze_results['clean_macro_f1']-haze_results['macro_f1']
output_path=results_dir/'haze_results.csv'
haze_results.to_csv(output_path,index=False)
print('Saved',output_path, 'Unavailable models:', missing)
display(haze_results.round(4))

## 5. Plot robustness curves and interpret cautiously

Plot **accuracy** and **macro-F1** against haze strength. If 13-band models degrade less than RGB, the evidence is only for the **defined synthetic transform** and should be evaluated across seeds. The default `(0,)` shows a pilot, not a three-seed final study. `07` remains optional according to the original project plan.

In [ ]:
import matplotlib.pyplot as plt
for model_name in haze_results['model'].unique():
    for profile in PROFILES:
        group=haze_results.query('model == @model_name and profile == @profile')
        if group.empty:
            continue
        fig,axes=plt.subplots(1,2,figsize=(11,4))
        for input_type in ('rgb','13band'):
            subset=group.query('input == @input_type')
            if subset.empty:
                continue
            mean=subset.groupby('strength')[['accuracy','macro_f1']].mean()
            axes[0].plot(mean.index,mean['accuracy'],marker='o',label=input_type)
            axes[1].plot(mean.index,mean['macro_f1'],marker='o',label=input_type)
        for ax,metric in zip(axes,('Accuracy','Macro-F1')):
            ax.set(title=f'{model_name}: {profile} haze | {metric}',
                   xlabel='Synthetic haze strength (unitless)',ylabel=metric)
            ax.grid(alpha=.25);ax.legend()
        fig.tight_layout()
        fig.savefig(figures_dir/f'haze_{model_name}_{profile}.png',dpi=150,bbox_inches='tight')
        plt.show()
print('Haze robustness files:',results_dir/'haze_results.csv')
print('Ran seeds:',RUN_SEEDS, 'Subset?', MAX_TEST_SAMPLES is not None)
print('CAVEAT: Synthetic stress test only. No real atmospheric haze claims.')

## 6. Optional-module conclusions and responsible reporting

- This analysis is an optional extension. The clean-model evaluation in `06` is the main evidence for the project question.
- Haze simulation uses linear mixing and manually assigned spectral factors; it is **not calibrated from real haze observations, optical depth, or atmospheric correction physics**.
- If checkpoint weights are missing, the notebook **skips** those models and states which are missing. Do not interpret a partial comparison as complete.
- The zero-strength replication check protects against inconsistent normalization, wrong checkpoint and wrong label alignment. **Do not remove that check**.
- For the final report, run on all held-out test patches with three seeds, report means and uncertainty, and describe the absence of real hazy EuroSAT labels.

**External methodological context:** [Remote Sensing, 2020, *Robust Single-Image Haze Removal Using Optimal Transmission Map and Adaptive Atmospheric Light*](https://www.mdpi.com/2072-4292/12/14/2233) discusses atmospheric scattering as an attenuation-plus-airlight model. This project only borrows its algebraic *form*; its particular band coefficients, bright reference and strengths are intentionally heuristic and **are not calibrated against the cited physical model**.